# 002 Feature Assembly

**Prerequisite**: run `002_run_feature_pipeline.py` to complete grid generation + data download + feature extraction.

This notebook is responsible for:
1. Interactive feature selection (edit `FEATURE_CONFIG`)
2. Loading grid + extracted data
3. Assembling `grid_gdf` + `all_node_features_col` according to configuration
4. Saving to `features/assembled/` for downstream graph construction

In [ ]:
import json
import pickle
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

FEATURES_DIR = Path('./results/intermediate/features')
GRID_DIR = FEATURES_DIR / 'grid'
EXTRACTED_DIR = FEATURES_DIR / 'extracted'
ASSEMBLED_DIR = FEATURES_DIR / 'assembled'
ASSEMBLED_DIR.mkdir(parents=True, exist_ok=True)

print(f'Feature directory: {FEATURES_DIR.resolve()}')

In [ ]:
# Region definitions (consistent with 002_run_feature_pipeline.py)
TRAIN_LOCATIONS = [
    'London',
    'TLH2', 'TLH3', 'TLJ1', 'TLF1', 'TLF2',
    'TLC1', 'TLC2', 'TLD6', 'TLG1', 'TLG2', 'TLE4',
]
TEST_LOCATIONS = ['TLH1', 'TLE3', 'TLD3', 'TLD4']
# TEST_LOCATIONS = []
ALL_LOCATIONS = TRAIN_LOCATIONS + TEST_LOCATIONS

print(f'Train regions: {len(TRAIN_LOCATIONS)}, Test regions: {len(TEST_LOCATIONS)}, Total: {len(ALL_LOCATIONS)}')

## Feature Selection Configuration

Edit `FEATURE_CONFIG` below to control the feature combination:
- `enabled`: whether to enable this feature group
- `type`: `"numerical"` = numerical column (merged into grid_gdf), `"array"` = high-dimensional array (saved separately)
- `columns`: which columns to select

In [ ]:
FEATURE_CONFIG = {
    "landuse": {
        "enabled": True,
        "type": "numerical",
        "source": "extracted/{ITL2}_landuse.npz",
        "columns": [
            "lu_residential_prop", "lu_commercial_prop",
            "lu_industrial_prop", "lu_agricultural_prop", "lu_others_prop"
        ]
    },
    "worldcover": {
        "enabled": True,
        "type": "numerical",
        "source": "extracted/{ITL2}_worldcover.npz",
        "columns": [
            "wc_built_up_ratio", "wc_agricultural_ratio", "wc_others_ratio"
        ]
    },
    "spectral": {
        "enabled": True,
        "type": "array",
        "source": "extracted/{ITL2}_spectral.npy",
        "columns": [
            "NDVI_mean", "NDVI_std", "NDVI_median",
            "NDBI_mean", "NDBI_std", "NDBI_median",
            "NDWI_mean", "NDWI_std", "NDWI_median",
            "BSI_mean", "BSI_std", "BSI_median",
            "UI_mean", "UI_std", "UI_median"
        ]
    },
    "qa": {
        "enabled": True,
        "type": "array",
        "source": "extracted/{ITL2}_qa.npy",
        "columns": [
            "q_residential", "q_commercial", "q_industrial",
            "q_agricultural", "q_others"
        ]
    }
}

# Show enabled feature groups
for name, cfg in FEATURE_CONFIG.items():
    status = 'ON' if cfg['enabled'] else 'OFF'
    print(f"  [{status}] {name}: type={cfg['type']}, {len(cfg['columns'])} columns")

In [ ]:
def load_grid(itl2: str):
    """Load the grid GeoDataFrame and step_size."""
    grid_path = GRID_DIR / f'{itl2}_grid.gpkg'
    meta_path = GRID_DIR / f'{itl2}_meta.json'

    grid_gdf = gpd.read_file(str(grid_path))
    with open(meta_path, 'r') as f:
        meta = json.load(f)
    return grid_gdf, meta['step_size_m']


def load_features(itl2: str, config: dict) -> dict:
    """Load feature data according to FEATURE_CONFIG."""
    features = {}
    for name, cfg in config.items():
        if not cfg['enabled']:
            continue

        source = cfg['source'].replace('{ITL2}', itl2)
        path = FEATURES_DIR / source

        if not path.exists():
            print(f"  [Warning] {name}/{itl2}: file not found {path}")
            continue

        if path.suffix == '.npy':
            features[name] = np.load(str(path))
        elif path.suffix == '.npz':
            loaded = np.load(str(path), allow_pickle=True)
            if cfg['type'] == 'numerical':
                if 'agg_data' in loaded:
                    data = loaded['agg_data']
                    col_names = list(loaded['agg_names'])
                else:
                    data = loaded['data']
                    col_names = list(loaded['columns'])
                features[name] = (data, col_names)
            else:
                if 'raw_data' in loaded:
                    features[name] = loaded['raw_data']
                elif 'data' in loaded:
                    features[name] = loaded['data']

    return features


def assemble_grid(grid_gdf: gpd.GeoDataFrame, features: dict, config: dict) -> gpd.GeoDataFrame:
    """Merge all feature columns from FEATURE_CONFIG into grid_gdf, and filter out region-level columns."""
    grid_gdf = grid_gdf.copy()

    for name, cfg in config.items():
        if not cfg['enabled'] or name not in features:
            continue

        expected_cols = cfg['columns']

        if cfg['type'] == 'numerical':
            # (data_array, col_names) tuple
            data, col_names = features[name]
            for col in expected_cols:
                if col in col_names:
                    grid_gdf[col] = data[:, col_names.index(col)]
                else:
                    print(f"  [Warning] column '{col}' not found in {name} data")

        elif cfg['type'] == 'array':
            # plain ndarray, column names come from config['columns']
            arr = features[name]
            for i, col in enumerate(expected_cols):
                if i < arr.shape[1]:
                    grid_gdf[col] = arr[:, i]
                else:
                    print(f"  [Warning] {name} array only has {arr.shape[1]} columns, skipping '{col}'")

    # Filter: keep only grid base columns + feature columns declared in FEATURE_CONFIG
    GRID_KEEP_COLS = ['geometry', 'index_region', 'ITL3', 'ITL2']
    all_feature_cols = []
    for cfg in config.values():
        if cfg['enabled']:
            all_feature_cols.extend(cfg['columns'])
    keep = GRID_KEEP_COLS + all_feature_cols
    grid_gdf = grid_gdf[[c for c in keep if c in grid_gdf.columns]]

    return grid_gdf

print('Loading functions defined')

In [ ]:
# Batch load + assemble
grid_data = {}   # {itl2: (grid_gdf, step_size_m)}
feat_data = {}   # {itl2: {name: array_or_tuple}}
errors = []

for location in ALL_LOCATIONS:
    try:
        grid_gdf, step_size_m = load_grid(location)
        features = load_features(location, FEATURE_CONFIG)
        grid_gdf = assemble_grid(grid_gdf, features, FEATURE_CONFIG)

        grid_data[location] = (grid_gdf, step_size_m)
        feat_data[location] = features

        # Verify point-count consistency
        n_grid = len(grid_gdf)
        feat_dims = []
        for name, fdata in features.items():
            if isinstance(fdata, np.ndarray):
                feat_dims.append(f"{name}={fdata.shape}")
                assert fdata.shape[0] == n_grid, f"{name} dimension mismatch: {fdata.shape[0]} vs {n_grid}"
            elif isinstance(fdata, tuple):
                feat_dims.append(f"{name}={fdata[0].shape}")
                assert fdata[0].shape[0] == n_grid, f"{name} dimension mismatch: {fdata[0].shape[0]} vs {n_grid}"

        print(f"  {location}: {n_grid} points, step={step_size_m}m, {', '.join(feat_dims)}")

    except Exception as e:
        errors.append((location, str(e)))
        print(f"  {location}: [Error] {e}")

print(f"\nSucceeded: {len(grid_data)}/{len(ALL_LOCATIONS)}")
if errors:
    print(f"Failed: {errors}")

In [ ]:
# Build feature schema — all feature columns are already in grid_gdf
numerical_col_names = []
categorical_col_members = {}

for name, cfg in FEATURE_CONFIG.items():
    if not cfg['enabled']:
        continue
    numerical_col_names.extend(cfg['columns'])

all_node_features_col = [numerical_col_names, categorical_col_members]

print(f'Feature columns ({len(numerical_col_names)}): {numerical_col_names}')
print(f'Categorical features: {categorical_col_members}')

In [ ]:
all_node_features_col

In [ ]:
# Save
saved_files = []

for location, (grid_gdf, step_size_m) in grid_data.items():
    out_path = ASSEMBLED_DIR / f'{location}_grid_points.pickle'
    with open(out_path, 'wb') as f:
        pickle.dump([grid_gdf, step_size_m], f)
    saved_files.append(out_path.name)

# all_node_features_col
with open(ASSEMBLED_DIR / 'all_node_features_col.pickle', 'wb') as f:
    pickle.dump(all_node_features_col, f)
saved_files.append('all_node_features_col.pickle')

# feature_schema.json
schema = {
    'numerical_col_names': numerical_col_names,
    'categorical_col_members': categorical_col_members,
    'locations': {
        'train': TRAIN_LOCATIONS,
        'test': TEST_LOCATIONS,
    },
    'region_stats': {
        loc: {'n_points': len(gdf), 'step_size_m': ssm}
        for loc, (gdf, ssm) in grid_data.items()
    },
}
with open(ASSEMBLED_DIR / 'feature_schema.json', 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)
saved_files.append('feature_schema.json')

print(f'Saved {len(saved_files)} files to {ASSEMBLED_DIR}')
for fn in saved_files:
    print(f'  - {fn}')

In [ ]:
# Validation statistics
rows = []
for location, (grid_gdf, step_size_m) in grid_data.items():
    nan_counts = {col: grid_gdf[col].isna().sum()
                  for col in numerical_col_names if col in grid_gdf.columns}
    total_nan = sum(nan_counts.values())
    rows.append({
        'region': location,
        'n_grid_points': len(grid_gdf),
        'n_columns': len(grid_gdf.columns),
        'step_size_m': step_size_m,
        'split': 'train' if location in TRAIN_LOCATIONS else 'test',
        'feature_nan_total': total_nan,
    })

summary_df = pd.DataFrame(rows)
print(f'Total grid points: {summary_df["n_grid_points"].sum():,}')
print(f'Train set points: {summary_df[summary_df["split"]=="train"]["n_grid_points"].sum():,}')
print(f'Test set points: {summary_df[summary_df["split"]=="test"]["n_grid_points"].sum():,}')
display(summary_df)

# Column validation
for location, (grid_gdf, step_size_m) in grid_data.items():
    bad_cols = [c for c in grid_gdf.columns
                if c in ('Demand (MVA)', 'Firm Capacity (MVA)', 'population',
                         'residential_percent', 'area', 'area_percent')]
    assert not bad_cols, f'{location}: region-level columns not filtered: {bad_cols}'
    print(f'{location} column validation passed: {list(grid_gdf.columns)}')

In [ ]:
grid_gdf

In [ ]:
# Compatibility check: confirm the pickle format is compatible with downstream usage
test_loc = list(grid_data.keys())[0]
test_path = ASSEMBLED_DIR / f'{test_loc}_grid_points.pickle'

with open(test_path, 'rb') as f:
    loaded = pickle.load(f)

assert isinstance(loaded, list) and len(loaded) == 2, 'pickle format error: expected [grid_gdf, step_size_m]'
assert isinstance(loaded[0], gpd.GeoDataFrame), 'the first element should be a GeoDataFrame'
assert isinstance(loaded[1], (int, float)), 'the second element should be numeric (step_size_m)'

print(f'Compatibility check passed: {test_loc}')
print(f'  grid_gdf: {loaded[0].shape}, columns={list(loaded[0].columns)}')
print(f'  step_size_m: {loaded[1]}')

# Validate all_node_features_col
with open(ASSEMBLED_DIR / 'all_node_features_col.pickle', 'rb') as f:
    loaded_features = pickle.load(f)
assert isinstance(loaded_features, list) and len(loaded_features) == 2
print(f'\nall_node_features_col check passed')
print(f'  numerical columns: {loaded_features[0]}')
print(f'  categorical columns: {loaded_features[1]}')